# 🌾 PlantVerse AI - Crop Recommendation System (Machine Learning & EDA)
**Project Title**: AI-Powered Smart Nursery & Plant Care Platform  
**Notebook Target**: Agronomy Soil N-P-K & Environmental Feature Analytics, Model Training, Evaluation & Visualizations.

---

### Notebook Highlights:
1. **Exploratory Data Analysis (EDA)**: Soil Nutrients ($N, P, K$), Temperature, Humidity, pH, and Rainfall distributions.
2. **Feature Correlation Heatmap**: Visualizing feature interactions across 19 crop species.
3. **Machine Learning Model Training**: Random Forest Classifier & Decision Tree Classifier comparison.
4. **Performance Evaluation**: Classification Report, Confusion Matrix, and Feature Importance Rankings.
5. **Real-time Crop Recommendation Engine**: Interactive inference function predicting optimal crops for given soil inputs.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette('crest')
print("✅ ML & Data Visualization Libraries Loaded Successfully!")


---
## Step 1: Dataset Loading & Overview
Loading synthetic dataset `crop_recommendation_dataset.csv` generated for 19 crop species (Basmati Rice, Wheat, Cotton, Sugarcane, Coffee, Tea, Banana, Mango, Apple, etc.).


In [ ]:
# Load dataset
df = pd.read_csv('crop_recommendation_dataset.csv')

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nHead of Dataset:")
display(df.head())

print("\nDataset Summary Statistics:")
display(df.describe())

print("\nCrop Value Counts:")
print(df['label'].value_counts())


---
## Step 2: Exploratory Data Analysis & Feature Visualizations
Analyzing how Nitrogen ($N$), Phosphorus ($P$), Potassium ($K$), Temperature, Humidity, pH, and Rainfall govern crop suitability.


In [ ]:
plt.figure(figsize=(16, 12))

# Subplot 1: Nitrogen Distribution by Top Crops
plt.subplot(2, 2, 1)
sns.boxplot(data=df, x='label', y='N', palette='viridis')
plt.title('Nitrogen (N) Requirement Across Crops', fontsize=12, fontweight='bold')
plt.xticks(rotation=60)

# Subplot 2: Rainfall Distribution by Crops
plt.subplot(2, 2, 2)
sns.boxplot(data=df, x='label', y='rainfall', palette='Blues_r')
plt.title('Rainfall (mm) Requirement Across Crops', fontsize=12, fontweight='bold')
plt.xticks(rotation=60)

# Subplot 3: Temperature vs Humidity Scatter
plt.subplot(2, 2, 3)
sns.scatterplot(data=df, x='temperature', y='humidity', hue='label', legend=False, alpha=0.8, palette='Spectral')
plt.title('Temperature vs Humidity Clusters', fontsize=12, fontweight='bold')

# Subplot 4: pH Distribution
plt.subplot(2, 2, 4)
sns.histplot(df['ph'], kde=True, color='teal', bins=25)
plt.title('Soil pH Distribution', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


### Feature Correlation Heatmap
Understanding feature collinearly between soil parameters and climate variables.


In [ ]:
plt.figure(figsize=(10, 7))
numeric_df = df.drop(columns=['label'])
corr = numeric_df.corr()
sns.heatmap(corr, annot=True, cmap='YlGnBu', fmt='.2f', linewidths=0.5)
plt.title('Agronomy Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.show()


---
## Step 3: Machine Learning Model Training (Random Forest & Decision Tree)
Splitting data into 80% Training and 20% Testing sets and training Scikit-Learn models.


In [ ]:
X = df[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size:  {X_test.shape[0]} samples")

# Train Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

# Train Decision Tree Classifier
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

rf_acc = accuracy_score(y_test, y_pred_rf)
dt_acc = accuracy_score(y_test, y_pred_dt)

print(f"🎯 Random Forest Accuracy: {rf_acc * 100:.2f}%")
print(f"🎯 Decision Tree Accuracy: {dt_acc * 100:.2f}%")


---
## Step 4: Model Evaluation & Feature Importance Analysis
Evaluating precision, recall, confusion matrix, and feature importances.


In [ ]:
print("================ CLASSIFICATION REPORT (Random Forest) ================")
print(classification_report(y_test, y_pred_rf))

# Confusion Matrix Plot
plt.figure(figsize=(12, 10))
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=rf_model.classes_, yticklabels=rf_model.classes_, cmap='Greens')
plt.title('Crop Recommendation Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Crop', fontweight='bold')
plt.ylabel('Actual Crop', fontweight='bold')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


### Feature Importance Bar Chart
Which agronomy metric has the highest influence on crop selection?


In [ ]:
importances = rf_model.feature_importances_
features = X.columns
feature_imp_df = pd.DataFrame({'Feature': features, 'Importance': importances}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=feature_imp_df, x='Importance', y='Feature', palette='emerald')
plt.title('Agronomy Feature Importances (Random Forest)', fontsize=13, fontweight='bold')
plt.xlabel('Importance Weight', fontweight='bold')
plt.tight_layout()
plt.show()


---
## Step 5: Real-Time Crop Recommendation Predictor Function
Simulating the PlantVerse AI recommendation engine API.


In [ ]:
def recommend_crop(N, P, K, temp, humidity, ph, rainfall):
    input_data = pd.DataFrame([[N, P, K, temp, humidity, ph, rainfall]], 
                              columns=['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall'])
    predicted_crop = rf_model.predict(input_data)[0]
    probs = rf_model.predict_proba(input_data)[0]
    confidence = max(probs) * 100
    
    print("🌾 ================= CROP RECOMMENDATION RESULT =================")
    print(f"Input Soil Parameters: N={N}, P={P}, K={K}, pH={ph}")
    print(f"Input Climate: Temp={temp}°C, Humidity={humidity}%, Rainfall={rainfall}mm")
    print(f"🌱 Recommended Optimal Crop : {predicted_crop.upper()}")
    print(f"📊 AI Confidence Score       : {confidence:.2f}%")
    print("================================================================")
    return predicted_crop

# Test Inference 1: Rice Conditions (High rainfall & high humidity)
recommend_crop(N=95, P=45, K=40, temp=24.5, humidity=85.0, ph=6.2, rainfall=220.0)

# Test Inference 2: Wheat Conditions (Moderate rainfall & cool temp)
recommend_crop(N=75, P=40, K=30, temp=18.0, humidity=60.0, ph=6.8, rainfall=75.0)

# Test Inference 3: Tea Conditions (Acidic soil & high rainfall)
recommend_crop(N=110, P=30, K=40, temp=21.0, humidity=82.0, ph=5.0, rainfall=200.0)
